# Cross-domain workflow: from material production to FEM material card

A manufacturer receives a batch of 316L austenitic stainless steel and wants to
use it in a finite element simulation. Before they can simulate, they need a
**material card**: a file the FEM solver reads to know how the material behaves
under load.

This notebook walks through all four steps in a single, traceable knowledge
graph, from the raw material to the exported solver card.

```
Step 1: Material Production           (PMDCo: ManufacturingProcess)
           |
           v
Step 2: Tensile Test (QA + data collection)   (TTO: TensileTest)
        ISO 6892-1 room-temperature test
        Yields: Rp0.2 = 285 MPa, Rm = 620 MPa, A = 50 %
           |
           v
Step 3: Constitutive Model Calibration        (OBI: ComputerSimulation)
        Hockett-Sherby fit to the flow curve
        sigma = sigma_sat - (sigma_sat - sigma_0) * exp(-c * eps_p^n)
           |
           v
Step 4: FEM Material Card Export
        Assemble elastic + plastic data; export as
        LS-Dyna *MAT_024 and Abaqus *PLASTIC keyword cards
```

Each step uses a different domain schema from this repository:

| Step | Schema | Ontology class |
|---|---|---|
| Material Production | `manufacturing/step/PMDCo/` | `pmdco:ManufacturingProcess` |
| Tensile Test | `characterization/tensile-test/TTO/` | `tto:TensileTest` |
| Model Calibration | `simulation/model-calibration/PMDCo/` | `obi:ComputerSimulation` |
| Material Card | `material-card/mechanical/PMDCo/` | `iao:DataSet` |

The `workflow/PMDCo/` schema ties them into a single navigable process record.


## Environment setup

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install jupyterlab jsonata-python rdflib pyshacl pyyaml
jupyter lab
```

In [ ]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

In [ ]:
import sys, json, math, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

# ── Folder layout ─────────────────────────────────────────────────────────────
# This notebook lives at:  schemas/workflow/PMDCo/docs/workflow_demo.ipynb
HERE      = pathlib.Path().resolve()                # .../workflow/PMDCo/docs/
WORKFLOW  = HERE.parent                             # .../workflow/PMDCo/
SCHEMAS   = WORKFLOW.parents[1]                     # .../schemas/

MANUF_STEP = SCHEMAS / "manufacturing"    / "step"              / "PMDCo"
TTO_DIR    = SCHEMAS / "characterization" / "tensile-test"      / "TTO"
CALIB      = SCHEMAS / "simulation"       / "model-calibration" / "PMDCo"
MATCARD    = SCHEMAS / "material-card"    / "mechanical"        / "PMDCo"
CHAR_BASE  = SCHEMAS / "characterization" / "step"              / "PMDCo"

def load_context(schema_dir):
    """Extract the @context block from a schema's schema.oold.yaml."""
    return yaml.safe_load((schema_dir / "specs" / "schema.oold.yaml").read_text())["@context"]

def transform(schema_dir, data):
    """Run the JSONata transform for a schema and return the OO-LD document."""
    expr = (schema_dir / "simplified" / "transform.jsonata").read_text()
    return Jsonata(expr).evaluate(data)

def parse_to_graph(context, oold_doc):
    """Parse an OO-LD document into an rdflib.Dataset and return a flat Graph."""
    ds = rdflib.Dataset()
    ds.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")
    g = rdflib.Graph()
    for s, p, o, _ in ds.quads():
        g.add((s, p, o))
    return g

print("\nSchema folders found:")
for label, path in [
    ("manufacturing/step/PMDCo",       MANUF_STEP),
    ("tensile-test/TTO",               TTO_DIR),
    ("model-calibration/PMDCo",        CALIB),
    ("material-card/mechanical/PMDCo", MATCARD),
]:
    print(f"  {'OK' if path.exists() else 'MISSING':6}  {label}")

## Part A: Building the four step records

Each step is described using its own domain schema, converted to RDF, and
validated independently. The four resulting graphs are later merged into one.

### Step 1: Material Production

The first step records that 316L rolled sheet (batch 1) was produced.
We note the rolling temperature and the final thickness as process conditions.
The output material gets an IRI that later steps will reference.

Schema used: `manufacturing/step/PMDCo/` (root class `pmdco:ManufacturingProcess`).

In [ ]:
prod_input = {
    "process_name": "316L Production, batch 1",
    "description":  "Melting, continuous casting, and hot rolling of 316L austenitic stainless steel. "
                    "Test coupons cut from batch 1 for QA tensile testing and material card generation.",
    "outputs": ["https://example.org/materials/316L-batch-1"],
    "conditions": [
        {"name": "Rolling Temperature", "value": 1150, "unit": "°C"},
        {"name": "Final Thickness",     "value": 6.0,  "unit": "mm"}
    ]
}

prod_oold = transform(MANUF_STEP, prod_input)
prod_iri  = prod_oold["id"]

print("OO-LD document for Step 1:")
print(json.dumps(prod_oold, indent=2))

In [ ]:
prod_graph = parse_to_graph(load_context(MANUF_STEP), prod_oold)
print(f"Manufacturing step graph: {len(prod_graph)} triples")
print(prod_graph.serialize(format="turtle"))

### Step 2: Tensile Test

A room-temperature tensile test following ISO 6892-1. The test consumes a
specimen cut from batch 1 and produces four mechanical property values:
yield strength (Rp0.2), tensile strength (Rm), elongation (A), and
reduction of area (Z).

Schema used: `characterization/tensile-test/TTO/` (root class `tto:TensileTest`).
This schema extends `characterization/step/PMDCo/` to add typed result nodes.
Two SHACL shape files are loaded: one for the base characterization step and one
for the tensile-test specialisation.

In [ ]:
tto_input = {
    "test_name":        "Tensile test 316L batch 1 QA",
    "specimen_iri":     "https://example.org/specimens/316L-batch-1-bar-1",
    "test_standard":    "ISO 6892-1",
    "strain_rate":      0.00025,
    "strain_rate_unit": "1/s",
    "temperature":      23,
    "results": [
        {"property": "YieldStrength",                     "value": 285, "unit": "MPa"},
        {"property": "TensileStrength",                   "value": 620, "unit": "MPa"},
        {"property": "PercentageElongationAfterFracture", "value": 50,  "unit": "%"},
        {"property": "PercentageReductionOfArea",         "value": 65,  "unit": "%"}
    ]
}

tto_oold = transform(TTO_DIR, tto_input)
tto_iri  = tto_oold["id"]

print("OO-LD document for Step 2:")
print(json.dumps(tto_oold, indent=2))

In [ ]:
tto_graph = parse_to_graph(load_context(TTO_DIR), tto_oold)
print(f"Tensile test graph: {len(tto_graph)} triples")

# Quick check: print the measured properties
TTO  = rdflib.Namespace("https://w3id.org/pmd/tto/")
OBI  = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
IAO  = rdflib.Namespace("http://purl.obolibrary.org/obo/IAO_")
UQDT = rdflib.Namespace("https://qudt.org/vocab/unit/")

print("\nMeasured properties:")
for result in tto_graph.objects(None, OBI["0000299"]):
    prop  = tto_graph.value(result, rdflib.RDF.type)
    val   = tto_graph.value(result, OBI["0001937"])
    unit  = tto_graph.value(result, IAO["0000039"])
    if prop and val:
        prop_short = str(prop).rsplit("/", 1)[-1]
        unit_short = str(unit).rsplit("/", 1)[-1] if unit else ""
        print(f"  {prop_short:<45}  {float(val):>8.1f}  {unit_short}")

### Step 3: Constitutive Model Calibration

The flow curve recorded during the tensile test (stress vs. plastic strain)
is fitted to a **Hockett-Sherby** model:

$$\\sigma(\\varepsilon_p) = \\sigma_{\\text{sat}} - (\\sigma_{\\text{sat}} - \\sigma_0) \\cdot e^{-c \\cdot \\varepsilon_p^n}$$

The four fitted parameters are stored as embedded result nodes in the graph
so they can be queried directly without parsing any external files.

Schema used: `simulation/model-calibration/PMDCo/`, extending `simulation/step/PMDCo/`.
The calibration node gets an IRI that the material card will reference as provenance.

In [ ]:
calib_input = {
    "step_name":   "Hockett-Sherby calibration 316L batch 1",
    "model_type":  "Hockett-Sherby",
    "description": "Levenberg-Marquardt fit of Hockett-Sherby flow curve to ISO 6892-1 tensile test data.",
    "inputs":  [f"https://example.org/base/{tto_iri}"],
    "conditions": [
        {"name": "Optimiser",    "unit": "Levenberg-Marquardt"},
        {"name": "Strain Range", "value": 0.5, "unit": "1"}
    ],
    "calibrated_parameters": [
        {"name": "sigma_sat", "value": 780.0, "unit": "MPa"},
        {"name": "sigma_0",   "value": 220.0, "unit": "MPa"},
        {"name": "c",         "value": 12.5},
        {"name": "n",         "value": 0.68}
    ]
}

calib_oold = transform(CALIB, calib_input)
calib_iri  = calib_oold["id"]

print("OO-LD document for Step 3:")
print(json.dumps(calib_oold, indent=2))

In [ ]:
calib_graph = parse_to_graph(load_context(CALIB), calib_oold)
print(f"Calibration graph: {len(calib_graph)} triples")

# Show the fitted parameters directly from the graph
QUDT = rdflib.Namespace("http://qudt.org/schema/qudt/")

print("\nCalibrated Hockett-Sherby parameters:")
for param in calib_graph.objects(None, OBI["0000299"]):
    name = calib_graph.value(param, rdflib.RDFS.label)
    val  = calib_graph.value(param, QUDT.value)
    unit = calib_graph.value(param, QUDT.hasUnit)
    if name and val:
        print(f"  {str(name):<12}  {float(val):>10.4f}  {str(unit) if unit else '(dimensionless)'}")

### Step 4a: Assemble the material card

The material card is a **data artefact** (not a process). It collects:
- elastic constants (density, Young's modulus, Poisson's ratio)
- discrete mechanical properties copied from the tensile test
- the Hockett-Sherby constitutive model parameters from the calibration

The `calibration_iri` field links the card back to the calibration node, creating
a machine-readable provenance trail from card to data.

Schema used: `material-card/mechanical/PMDCo/` (root class `iao:DataSet`).

In [ ]:
card_input = {
    "card_name":    "316L stainless steel, Hockett-Sherby v1",
    "material_iri": "https://example.org/materials/316L-batch-1",
    "description":  "Mechanical material card for 316L austenitic SS batch 1. "
                    "Elastic constants from literature; plastic data from ISO 6892-1 tensile test.",
    "density":        {"value": 7.93,  "unit": "g/cm³"},
    "youngs_modulus": {"value": 193.0, "unit": "GPa"},
    "poissons_ratio": {"value": 0.29},
    "mechanical_properties": [
        {"type": "YieldStrength",                     "value": 285.0, "unit": "MPa"},
        {"type": "TensileStrength",                   "value": 620.0, "unit": "MPa"},
        {"type": "PercentageElongationAfterFracture", "value": 50.0,  "unit": "%"}
    ],
    "constitutive_model": {
        "model_type":      "Hockett-Sherby",
        "calibration_iri": f"https://example.org/base/{calib_iri}",
        "parameters": [
            {"name": "sigma_sat", "value": 780.0, "unit": "MPa"},
            {"name": "sigma_0",   "value": 220.0, "unit": "MPa"},
            {"name": "c",         "value": 12.5},
            {"name": "n",         "value": 0.68}
        ]
    }
}

card_oold = transform(MATCARD, card_input)
card_iri  = card_oold["id"]

print("OO-LD document for material card:")
print(json.dumps(card_oold, indent=2))

In [ ]:
card_graph = parse_to_graph(load_context(MATCARD), card_oold)
print(f"Material card graph: {len(card_graph)} triples")
print(card_graph.serialize(format="turtle"))

## Part B: Assembling the workflow

The workflow schema (`workflow/PMDCo/`) creates a lightweight container node
(`bfo:Process`) that lists all four steps and connects them in sequence.
Each step's `instance_iri` points to the detailed record we just built above.

Step 4 (FEM Material Card Export) does not have its own dedicated schema.
It is recorded directly in the workflow with a description and inline parameters
that describe the export target. The actual export happens in Part E below.

In [ ]:
workflow_input = {
    "workflow_name": "QA-to-FEM material card workflow, 316L batch 1",
    "description":   "Four-step cross-domain workflow: material production to calibrated FEM material card.",
    "steps": [
        {
            "label":        "Material Production, 316L batch 1",
            "step_type":    "pmdco:PMD_0000029",
            "description":  "Hot-rolled 316L sheet, batch 1.",
            "instance_iri": f"https://example.org/base/{prod_iri}"
        },
        {
            "label":        "Tensile Test, 316L batch 1 QA",
            "step_type":    "obi:0000070",
            "description":  "ISO 6892-1 room-temperature tensile test. Yields Rp0.2, Rm, A, Z.",
            "instance_iri": f"https://example.org/base/{tto_iri}"
        },
        {
            "label":        "Hockett-Sherby Calibration, 316L batch 1",
            "step_type":    "obi:0000471",
            "description":  "Hockett-Sherby flow curve fit to ISO 6892-1 data.",
            "instance_iri": f"https://example.org/base/{calib_iri}"
        },
        {
            "label":       "FEM Material Card Export, LS-Dyna / Abaqus",
            "description": "Query calibrated parameters from the knowledge graph and export "
                           "a syntactic material card for LS-Dyna (*MAT_024) and Abaqus (*PLASTIC).",
            "conditions": [
                {"name": "Target Solver", "unit": "LS-Dyna R14 / Abaqus 2023"},
                {"name": "Card Format",   "unit": "*MAT_024 / *PLASTIC"}
            ]
        }
    ]
}

wf_oold = transform(WORKFLOW, workflow_input)
print("Workflow OO-LD (step summary):")
for s in wf_oold["has_part"]:
    pre = s.get("preceded_by", ["(first)"])
    print(f"  {s['label']}")
    print(f"    preceded_by: {pre}")

In [ ]:
wf_graph = parse_to_graph(load_context(WORKFLOW), wf_oold)
print(f"Workflow graph: {len(wf_graph)} triples")

## Part C: One combined knowledge graph

All five graphs (manufacturing, tensile test, calibration, material card, workflow)
are merged into a single graph. In a real deployment each graph would be stored as
a named graph in a triple store; here we flatten everything for easy querying.

The combined graph is also SHACL-validated against the workflow shape.

In [ ]:
combined = rdflib.Graph()
for g in [prod_graph, tto_graph, calib_graph, card_graph, wf_graph]:
    combined += g

print(f"Combined graph: {len(combined)} triples")

out_ttl = HERE / "output_workflow_combined.ttl"
out_ttl.write_text(combined.serialize(format="turtle"))
print(f"Written to {out_ttl}")

In [ ]:
shapes = rdflib.Graph().parse(str(WORKFLOW / "specs" / "shape.ttl"))
# Wrap combined in a Dataset for pyshacl
ds_for_shacl = rdflib.Dataset()
for t in combined:
    ds_for_shacl.add(t)

conforms, results_graph, _ = pyshacl.validate(ds_for_shacl, shacl_graph=shapes, inference="rdfs")
print(f"Workflow SHACL conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for r in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        print(f"  x {results_graph.value(r, SH.resultMessage)}")

## Part D: Cross-domain SPARQL queries

SPARQL is the query language for RDF graphs. The queries below cross domain
boundaries, navigating from the workflow container all the way to individual
parameter values. This is only possible because all four step records share
the same graph and the same ontology mappings.

You do not need to understand SPARQL to benefit from this section.
Read the comments and the printed output.

In [ ]:
# ── Query 1: workflow steps and their ordering ────────────────────────────────
SPARQL_STEPS = """
PREFIX bfo:     <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?stepLabel ?type ?predecessor
WHERE {
  ?workflow a bfo:0000015 ; bfo:0000051 ?step .
  ?step rdfs:label ?stepLabel .
  OPTIONAL { ?step a ?type . FILTER(?type != bfo:0000015) }
  OPTIONAL { ?step bfo:0000062 ?pred .
             ?pred rdfs:label ?predecessor . }
}
ORDER BY ?step
"""

print(f"{'Step':<55} {'Domain class':<22} {'Preceded by'}")
print("-" * 110)
seen = {}
for r in combined.query(SPARQL_STEPS):
    label = str(r.stepLabel)
    typ   = str(r.type).rsplit("/", 1)[-1].rsplit("#", 1)[-1] if r.type else "BFO:Process"
    pred  = str(r.predecessor) if r.predecessor else "(first step)"
    if label not in seen:
        seen[label] = True
        print(f"{label:<55} {typ:<22} {pred}")
    else:
        print(f"{'':55} {'':22} {pred}")

In [ ]:
# ── Query 2: tensile test results in the combined graph ───────────────────────
SPARQL_RESULTS = """
PREFIX tto:   <https://w3id.org/pmd/tto/>
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>

SELECT ?propertyClass ?value ?unit
WHERE {
  ?test  a tto:TensileTest ; obi:0000299 ?result .
  ?result a ?propertyClass ; obi:0001937 ?value ; iao:0000039 ?unit .
}
ORDER BY ?propertyClass
"""

print("Tensile test results:")
print(f"  {'Property':<45}  {'Value':>8}  Unit")
for r in combined.query(SPARQL_RESULTS):
    prop  = str(r.propertyClass).rsplit("/", 1)[-1]
    unit  = str(r.unit).rsplit("/", 1)[-1]
    print(f"  {prop:<45}  {float(r.value):>8.1f}  {unit}")

In [ ]:
# ── Query 3: constitutive model parameters in the combined graph ──────────────
SPARQL_PARAMS = """
PREFIX obi:     <http://purl.obolibrary.org/obo/OBI_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt:    <http://qudt.org/schema/qudt/>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?modelType ?paramName ?value ?unit
WHERE {
  ?calib a obi:0000471 ;
         dcterms:type ?modelType ;
         obi:0000299  ?param .
  ?param rdfs:label ?paramName ;
         qudt:value  ?value .
  OPTIONAL { ?param qudt:hasUnit ?unit . }
}
ORDER BY ?paramName
"""

rows = list(combined.query(SPARQL_PARAMS))
if rows:
    model = str(rows[0].modelType)
    print(f"Constitutive model: {model}\n")
    print(f"  {'Parameter':<12}  {'Value':>12}  Unit")
    for r in rows:
        unit = str(r.unit) if r.unit else "(dimensionless)"
        print(f"  {str(r.paramName):<12}  {float(r.value):>12.4f}  {unit}")

## Part E: FEM material card export

This is where the workflow pays off. We query the combined graph for all the
data an FEM solver needs and produce ready-to-use keyword cards.

**No files are read here. Everything comes from the graph.**

The Hockett-Sherby parameters from Step 3 are used to generate a tabulated
flow curve (true stress vs. plastic strain), which both solvers accept:

$$\\sigma(\\varepsilon_p) = \\sigma_{\\text{sat}} - (\\sigma_{\\text{sat}} - \\sigma_0) \\cdot e^{-c \\cdot \\varepsilon_p^n}$$

The helper functions below handle all the formatting details so the export
cells stay short. They are defined once and can be ignored unless you want
to customise the output.

In [ ]:
# ── Helper functions for the FEM export ──────────────────────────────────────
# These are defined here so the export cells below stay short and readable.
# You do not need to modify these unless you want to change the output format.

PMDCO_NS = rdflib.Namespace("https://w3id.org/pmd/co/")
QUDT_NS  = rdflib.Namespace("http://qudt.org/schema/qudt/")

def extract_card_data(graph):
    """Read density, Young's modulus, Poisson's ratio, and model parameters
    from the combined RDF graph. Returns (rho_gcm3, E_GPa, nu, params_dict)."""

    def scalar(prop_iri):
        for node in graph.objects(None, prop_iri):
            v = graph.value(node, QUDT_NS.value)
            if v:
                return float(v)
        return None

    rho   = scalar(PMDCO_NS["PMD_0000025"])   # Density
    E_gpa = scalar(PMDCO_NS["PMD_0000039"])   # YoungModulus
    nu    = scalar(PMDCO_NS["PMD_0000040"])   # PoissonRatio

    params = {}
    for r in graph.query("""
        PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX qudt: <http://qudt.org/schema/qudt/>
        SELECT ?name ?value WHERE {
          ?calib a obi:0000471 ; obi:0000299 ?p .
          ?p rdfs:label ?name ; qudt:value ?value .
        }
    """):
        params[str(r.name)] = float(r.value)

    return rho, E_gpa, nu, params


def build_flow_curve(params, n_points=21):
    """Evaluate the Hockett-Sherby equation at n_points plastic strain values.
    Returns a list of (plastic_strain, stress_MPa) pairs."""
    s, s0, c, n = params["sigma_sat"], params["sigma_0"], params["c"], params["n"]
    return [(i * 0.025,
             s0 if i == 0 else s - (s - s0) * math.exp(-c * (i * 0.025) ** n))
            for i in range(n_points)]


def as_lsdyna_card(rho_gcm3, E_GPa, nu, flow_curve):
    """Return an LS-Dyna *MAT_024 keyword string.
    Unit system: tonne / mm / s / N / MPa."""
    rho = rho_gcm3 * 1e-9        # g/cm³  → t/mm³
    E   = E_GPa   * 1e3          # GPa    → MPa
    header = (
        f"$$ LS-Dyna *MAT_PIECEWISE_LINEAR_PLASTICITY  (units: t/mm³, MPa)\n"
        f"*MAT_PIECEWISE_LINEAR_PLASTICITY\n"
        f"$# MID        RHO         E           PR\n"
        f"         1  {rho:.4e}  {E:.1f}       {nu:.3f}\n\n"
        f"*DEFINE_CURVE\n$$ plastic strain vs. true stress (MPa)\n         1"
    )
    rows = "\n".join(f"{eps:>12.4f}  {sig:>12.2f}" for eps, sig in flow_curve)
    return header + "\n" + rows + "\n\n*END"


def as_abaqus_card(rho_gcm3, E_GPa, nu, flow_curve):
    """Return an Abaqus *MATERIAL keyword string.
    Unit system: kg / m / s / Pa."""
    rho = rho_gcm3 * 1000.0      # g/cm³  → kg/m³
    E   = E_GPa   * 1e9          # GPa    → Pa
    header = (
        f"** Abaqus *MATERIAL  (units: kg/m³, Pa)\n"
        f"*MATERIAL, NAME=316L_HS_v1\n"
        f"*DENSITY\n{rho:.1f},\n"
        f"*ELASTIC\n{E:.6e}, {nu:.3f}\n"
        f"*PLASTIC\n** yield stress (Pa),  plastic strain"
    )
    rows = "\n".join(f"{sig * 1e6:>14.2f},  {eps:.4f}" for eps, sig in flow_curve)
    return header + "\n" + rows

In [ ]:
# Extract everything needed from the combined graph
rho, E_gpa, nu, hs_params = extract_card_data(combined)

print(f"Density          : {rho} g/cm³")
print(f"Young's modulus  : {E_gpa} GPa")
print(f"Poisson's ratio  : {nu}")
print(f"Model parameters : {hs_params}")

# Build the tabulated flow curve (21 points, eps_p = 0 to 0.5)
table = build_flow_curve(hs_params)
print(f"\nFlow curve: {len(table)} points, "
      f"sigma ranges from {table[0][1]:.0f} to {table[-1][1]:.0f} MPa")

### LS-Dyna (*MAT_024)

`*MAT_PIECEWISE_LINEAR_PLASTICITY` accepts a tabulated flow curve directly.
The card below uses the tonne / mm / MPa unit system common in automotive FEM.

In [ ]:
lsdyna_card = as_lsdyna_card(rho, E_gpa, nu, table)
print(lsdyna_card)

### Abaqus (*PLASTIC)

The `*PLASTIC` block expects pairs of (yield stress in Pa, plastic strain).
The same tabulated curve is used; only the unit system differs.

In [ ]:
abaqus_card = as_abaqus_card(rho, E_gpa, nu, table)
print(abaqus_card)

In [ ]:
# Save both cards to files
(HERE / "316L_HS_v1.k").write_text(lsdyna_card)
(HERE / "316L_HS_v1.inp").write_text(abaqus_card)
print("Saved  316L_HS_v1.k   (LS-Dyna)")
print("Saved  316L_HS_v1.inp (Abaqus)")

## Summary

| Section | What happened |
|---|---|
| Part A, step 1 | Manufacturing process recorded with PMDCo schema; output material IRI created |
| Part A, step 2 | Tensile test recorded with TTO schema; Rp0.2, Rm, A, Z stored as typed RDF nodes |
| Part A, step 3 | Hockett-Sherby calibration recorded; sigma_sat, sigma_0, c, n stored in the graph |
| Part A, step 4 | Material card assembled as IAO:DataSet; elastic constants + constitutive model embedded |
| Part B | Workflow container (BFO:Process) created; four steps linked by `preceded_by` |
| Part C | All five graphs merged; SHACL validation passed |
| Part D | Cross-domain SPARQL: workflow to tensile results to model parameters |
| Part E | Hockett-Sherby flow curve tabulated; LS-Dyna and Abaqus keyword cards written to file |

The entire export (Part E) reads from the RDF graph; no JSON files and no
hard-coded constants. If the calibration or the elastic constants change, re-run
all cells and new solver cards are generated automatically.


## Adapting to your own material

1. Replace the `_input` dictionaries in Part A with your own data.
2. Re-run all cells.
3. Collect the exported `.k` and `.inp` files.

To use a different constitutive model, change `model_type` and
`calibrated_parameters` in the calibration step, then update the flow-curve
equation in Part E accordingly.


## Further reading

- [Workflow schema README](../README.md)
- [Manufacturing Step (PMDCo)](../../../manufacturing/step/PMDCo/README.md)
- [Tensile Test (TTO)](../../../characterization/tensile-test/TTO/README.md)
- [Constitutive Model Calibration (PMDCo)](../../../simulation/model-calibration/PMDCo/README.md)
- [Mechanical Material Card (PMDCo)](../../../material-card/mechanical/PMDCo/README.md)
- [OO-LD primer](../../../../docs/oold-primer.md)